In [12]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(":memory:")

players_fresh_stats = pd.read_csv('../data/players_fresh_stats.csv')
appearances = pd.read_csv('../data/appearances.csv')
players = pd.read_csv('../data/players.csv')
player_valuations = pd.read_csv('../data/player_valuations.csv')
players_fresh_stats.to_sql("players_fresh_stats", conn, index=False)
appearances.to_sql("appearances", conn, index=False)
players.to_sql("players", conn, index=False)
player_valuations.to_sql("player_valuations", conn, index=False)
players_fresh_final = pd.read_csv('../data/players_fresh_final.csv')
players_fresh_final.to_sql("players_fresh_final", conn, index=False)

def load_query(filepath):
    with open(filepath) as f:
        return f.read()


# Query 1: Top scorers among fresh players

**Result:** Harry Kane leads with 414 goals. This differs from 03_features.ipynb, where on the full dataset Ronaldo (434), Messi (458), and Lewandowski (528) were the top scorers.
Ronaldo, Messi, and Lewandowski are excluded from players_fresh_stats because their last Transfermarkt valuation is stale  (>365 days old) — they were filtered out in 02_data_check.ipynb, before this JOIN ever runs. This top scorers list therefore reflects only actively-tracked players, not the all-time top scorers in the dataset

In [13]:
q1 = load_query('../sql/top_scorers.sql')
print(pd.read_sql_query(q1, conn))

   player_id                 name  total_goals  total_appearances
0     132098           Harry Kane          382                528
1     148455        Mohamed Salah          297                547
2     105521        Ciro Immobile          271                511
3      96341        Romelu Lukaku          270                547
4     125781    Antoine Griezmann          244                601
5      93720  Alexandre Lacazette          239                505
6     418560       Erling Haaland          237                259
7      68863         Mauro Icardi          231                433
8      72522         Luuk de Jong          222                524
9      91845        Heung-min Son          208                553


# Query 2: goals and assists per 90 minutes

**Result:** at a 2000-minute threshold (22 full matches), the top is led by Haaland, Gyökeres, and Franculino. Limitation is that per-90 rate doesn't take into account opponent strength, exact role within a position and penalty goals. Two players with the same goals_per_90 could have very different underlying skill profiles

In [14]:
q2 = load_query('../sql/stats_per_90.sql')
print(pd.read_sql(q2, conn))

              name  total_goals  total_assists  total_minutes  goals_per_90  \
0   Erling Haaland          237             46          21056         1.013   
1  Viktor Gyökeres           93             21           8774         0.954   
2       Franculino           45             13           4788         0.846   
3       Harry Kane          382             90          42703         0.805   
4   Jefté Betancor           22              1           2597         0.762   
5       Jhon Durán           19              3           2293         0.746   
6     Ricardo Pepi           45             11           5530         0.732   
7   Ayoub El Kaabi           85             12          10705         0.715   
8   Victor Osimhen          147             33          18800         0.704   
9     Paul Onuachu          163             33          21211         0.692   

   assists_per_90  
0           0.197  
1           0.215  
2           0.244  
3           0.190  
4           0.035  
5         

# Query 3: Ranking players within their position (RANK vs DENSE_RANK)

**RANK vs DENSE_RANK:** at rows 28-30, three Attack players share the same goals_per_90 (0.592). RANK() assigns them 29, 29, 29 and then jumps the next player to 32, skipping 30 and 31 to account for the tie.DENSE_RANK() assigns them 29, 29, 29, 30 — no gap, treating the tie as one shared rank level.
This view only shows the Attack position. Metrics based on goals are not meaningful for Goalkeepers and Defenders

In [15]:
q3 = load_query('../sql/ranked_stats.sql')
print(pd.read_sql(q3, conn))

                   name  total_goals  total_assists  total_minutes position  \
0        Erling Haaland          237             46          21056   Attack   
1       Viktor Gyökeres           93             21           8774   Attack   
2            Franculino           45             13           4788   Attack   
3            Harry Kane          382             90          42703   Attack   
4        Jefté Betancor           22              1           2597   Attack   
5            Jhon Durán           19              3           2293   Attack   
6          Ricardo Pepi           45             11           5530   Attack   
7        Ayoub El Kaabi           85             12          10705   Attack   
8        Victor Osimhen          147             33          18800   Attack   
9          Paul Onuachu          163             33          21211   Attack   
10        Promise David           20              4           2653   Attack   
11         Mauro Icardi          231             63 

# Query 4: Market value change over time (Cristiano Ronaldo, player_id=8198)

**Result:** Ronaldo's value has steady growth through 2013-2014 (peaking around €120M), then a gradual decline from 2015-2016. This matches the well-known age/value curve in football, where market value typically peaks around age 24-27 and declines afterward regardless of continued performance

In [16]:
q4 = load_query('../sql/value_change.sql')
print(pd.read_sql(q4, conn))

          date               name  market_value_in_eur  value_change
0   2004-10-04  Cristiano Ronaldo             18000000             0
1   2005-02-11  Cristiano Ronaldo             22799999       4799999
2   2005-10-18  Cristiano Ronaldo             26500000       3700001
3   2006-06-14  Cristiano Ronaldo             25000000      -1500000
4   2007-04-25  Cristiano Ronaldo             34000000       9000000
5   2008-02-04  Cristiano Ronaldo             50000000      16000000
6   2008-04-08  Cristiano Ronaldo             55000000       5000000
7   2008-07-04  Cristiano Ronaldo             60000000       5000000
8   2009-07-22  Cristiano Ronaldo             70000000      10000000
9   2009-12-03  Cristiano Ronaldo             70000000             0
10  2010-01-07  Cristiano Ronaldo             75000000       5000000
11  2010-04-12  Cristiano Ronaldo             75000000             0
12  2010-08-27  Cristiano Ronaldo             90000000      15000000
13  2011-02-04  Cristiano Ronaldo 